# 01 Data Prep

Loads all NHAMCS ED SAS files (2015–2022), performs **all** sentinel/format cleaning in one place,
and writes model-ready CSVs to `data/`.

**Outputs:**
- `data/eda_df.csv` — full cleaned dataset; includes PATWT, REGION, MSA, SETTYPE
- `data/model_A_arrival_dataset.csv` — features available at time of arrival + PATWT
- `data/model_B_retrospective_dataset.csv` — all features including operational vars + PATWT

**Rules:**
- PATWT (patient visit weight) is in every output. The CDC documentation requires it for all NHAMCS estimates.
- All sentinel values (-9, -8, 998, 999, etc.) and the TEMPF ×10 format bug are fixed here. Downstream notebooks must not repeat this cleaning.

In [1]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

data_folder = Path('../data')
sas_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.sas7bdat')])
print(f'Found {len(sas_files)} SAS files:')
for f in sas_files:
    print(' -', f)

dfs = []
for file in sas_files:
    file_path = data_folder / file
    tmp = pd.read_sas(file_path, format='sas7bdat', encoding='latin1')
    if 'YEAR' not in tmp.columns:
        match = re.search(r'(20\d{2}|15|16|17|18)$', file.replace('.sas7bdat', '').replace('_sas', ''))
        if match:
            y = int(match.group(1))
            if y < 100:
                y = 2000 + y
            tmp['YEAR'] = y
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True, sort=False)
print(f'\nCombined shape: {df.shape}')

# Confirm the columns we need most are present
for col in ['PATWT', 'REGION', 'MSA', 'SETTYPE', 'WAITTIME', 'IMMEDR']:
    status = 'OK' if col in df.columns else 'MISSING'
    print(f'  {col}: {status}')

Found 6 SAS files:
 - ed2015-sas.sas7bdat
 - ed2016_sas.sas7bdat
 - ed2017_sas.sas7bdat
 - ed2018_sas.sas7bdat
 - ed2021_sas.sas7bdat
 - ed2022_sas.sas7bdat



Combined shape: (109760, 1061)
  PATWT: OK
  REGION: OK
  MSA: OK
  SETTYPE: OK
  WAITTIME: OK
  IMMEDR: OK


In [2]:
# ── 1. Column selection ────────────────────────────────────────────────────────
# REGION and MSA are required for weighted analysis and nonresponse adjustment.
# PATWT and EDWT are required by the CDC for any NHAMCS estimate.
# SETTYPE identifies the ESA type (general, fast-track, pediatric, etc.).
candidate_cols = [
    'WAITTIME', 'ARRTIME', 'VMONTH', 'VDAYR', 'YEAR',
    'PATWT', 'EDWT', 'REGION', 'MSA', 'SETTYPE',
    'AGE', 'SEX', 'RACEUN', 'ETHUN', 'IMMEDR', 'PAINSCALE',
    'LOV', 'ADMIT', 'FASTTRAK', 'OBSCLIN', 'BOARD', 'BOARDED', 'BOARDHOS',
    'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT',
    'BEDREG', 'IMBED', 'BEDCZAR',
    'MRI', 'XRAY', 'CT', 'CTCONTRAST', 'ULTRASOUND', 'ANYIMAGE', 'LABTEST', 'TOTPROC',
]
existing_cols = [c for c in candidate_cols if c in df.columns]
missing_cols  = [c for c in candidate_cols if c not in df.columns]
if missing_cols:
    print(f'WARNING — not found in raw data: {missing_cols}')

eda_df = df[existing_cols].copy()
print(f'Selected {len(existing_cols)} columns → shape {eda_df.shape}')

# ── 2. Force numeric on every column except ARRTIME ────────────────────────────
non_numeric = {'ARRTIME'}
for c in existing_cols:
    if c not in non_numeric:
        eda_df[c] = pd.to_numeric(eda_df[c], errors='coerce')

# ── 3. Parse ARRTIME → ARRIVAL_HOUR ───────────────────────────────────────────
def _arrtime_to_hour(val):
    try:
        s = val if isinstance(val, str) else val.decode('latin1') if isinstance(val, (bytes, bytearray)) else str(val)
        digits = ''.join(ch for ch in s if ch.isdigit())
        if len(digits) == 3:
            digits = '0' + digits
        if len(digits) != 4:
            return np.nan
        hh, mm = int(digits[:2]), int(digits[2:])
        return hh if (0 <= hh <= 23 and 0 <= mm <= 59) else np.nan
    except Exception:
        return np.nan

eda_df['ARRIVAL_HOUR'] = eda_df['ARRTIME'].apply(_arrtime_to_hour)

# ── 4. Sentinel value cleaning (all codes defined once here) ──────────────────
# NHAMCS uses -9/-8/-7 for unknown/not-applicable and high codes (98/99, 998/999)
# for "not answered" or "not applicable". These must become NaN before any analysis.
sentinel_map = {
    'PAINSCALE':  {-9, -8, 99},
    'BPSYS':      {-9, -8, 998, 999},
    'BPDIAS':     {-9, -8, 998, 999},
    'TEMPF':      {-9, -8, 998, 999, 9980, 9989, 9990, 9999},
    'PULSE':      {-9, -8, 998, 999},
    'RESPR':      {-9, -8, 98, 99, 998, 999},
    'POPCT':      {-9, -8, 998, 999},
    'IMMEDR':     {-9, -8, 8, 9, 98, 99},
    'SEX':        {-9, 9, 99},
    'RACEUN':     {-9, 9, 99},
    'ETHUN':      {-9, 9, 99},
    'LOV':        {-9, -8},
    'BOARD':      {-9, -8},
    'BOARDHOS':   {-9, -8},
    'BOARDED':    {-9, -8},
    'ADMIT':      {-9, -8},
    'FASTTRAK':   {-9, -8},
    'OBSCLIN':    {-9, -8},
    'BEDREG':     {-9, -8},
    'IMBED':      {-9, -8},
    'BEDCZAR':    {-9, -8},
    'MRI':        {-9, -8},
    'XRAY':       {-9, -8},
    'CT':         {-9, -8},
    'CTCONTRAST': {-9, -8},
    'ULTRASOUND': {-9, -8},
    'ANYIMAGE':   {-9, -8},
    'LABTEST':    {-9, -8},
    'TOTPROC':    {-9, -8},
}
for col, bad_vals in sentinel_map.items():
    if col in eda_df.columns:
        mask = eda_df[col].isin(bad_vals)
        if mask.any():
            eda_df.loc[mask, col] = np.nan

# ── 5. TEMPF format fix ────────────────────────────────────────────────────────
# NHAMCS stores temperature as integer ×10 (e.g. 986 = 98.6 °F).
# Any value > 120 after sentinel removal is physiologically impossible in °F,
# so it must be a ×10 encoded value. Then null anything still outside 85–115 °F.
if 'TEMPF' in eda_df.columns:
    mask_10x = eda_df['TEMPF'] > 120
    eda_df.loc[mask_10x, 'TEMPF'] = eda_df.loc[mask_10x, 'TEMPF'] / 10.0
    print(f'TEMPF: divided {mask_10x.sum()} ×10 values by 10')
    mask_bad = ~eda_df['TEMPF'].between(85, 115) & eda_df['TEMPF'].notna()
    eda_df.loc[mask_bad, 'TEMPF'] = np.nan
    print(f'TEMPF: nulled {mask_bad.sum()} remaining out-of-range values')

# ── 6. Range / validity filters ───────────────────────────────────────────────
eda_df = eda_df[eda_df['WAITTIME'].between(0, 480)].copy()
eda_df.loc[~eda_df['VMONTH'].between(1, 12),      'VMONTH']       = np.nan
eda_df.loc[~eda_df['VDAYR'].between(1, 7),        'VDAYR']        = np.nan
eda_df.loc[~eda_df['ARRIVAL_HOUR'].between(0, 23), 'ARRIVAL_HOUR'] = np.nan

# ── 7. Drop exact duplicates ───────────────────────────────────────────────────
before = len(eda_df)
eda_df = eda_df.drop_duplicates().reset_index(drop=True)
print(f'Dropped {before - len(eda_df)} duplicate rows → shape {eda_df.shape}')

# ── 8. Data quality summary ────────────────────────────────────────────────────
print('\nMissing % for key columns (after cleaning):')
key_cols = ['WAITTIME', 'PATWT', 'REGION', 'MSA', 'IMMEDR', 'ARRIVAL_HOUR',
            'TEMPF', 'PAINSCALE', 'FASTTRAK', 'OBSCLIN', 'BOARD', 'LOV']
for c in key_cols:
    if c in eda_df.columns:
        pct = eda_df[c].isna().mean() * 100
        print(f'  {c:<14} {pct:5.1f}% missing')

print(f'\nPATWT range: {eda_df["PATWT"].min():.0f} – {eda_df["PATWT"].max():.0f}')
print(f'TEMPF range after fix: {eda_df["TEMPF"].min():.1f} – {eda_df["TEMPF"].max():.1f}')

WARNING — not found in raw data: ['CT', 'ULTRASOUND', 'LABTEST']
Selected 37 columns → shape (109760, 37)


TEMPF: divided 101948 ×10 values by 10
TEMPF: nulled 1 remaining out-of-range values
Dropped 40 duplicate rows → shape (91811, 38)

Missing % for key columns (after cleaning):
  WAITTIME         0.0% missing
  PATWT            0.0% missing
  REGION           0.0% missing
  MSA              0.0% missing
  IMMEDR          20.7% missing
  ARRIVAL_HOUR     0.0% missing
  TEMPF            5.3% missing
  PAINSCALE       33.2% missing
  FASTTRAK         6.9% missing
  OBSCLIN          6.0% missing
  BOARD            6.2% missing
  LOV             34.7% missing

PATWT range: 40 – 57926
TEMPF range after fix: 86.5 – 109.0


In [ ]:
# ── Output column sets ─────────────────────────────────────────────────────────
# PATWT, REGION, MSA are included in every model output.
# SETTYPE is excluded — it is constant (all=3, General ED) and carries zero information.
# model_A: only features available at time of arrival (no post-visit outcomes).
# model_B: full feature set including operational/resource-use vars for bottleneck analysis.

arrival_cols = [c for c in [
    'WAITTIME', 'YEAR', 'PATWT', 'REGION', 'MSA',
    'ARRTIME', 'ARRIVAL_HOUR', 'VMONTH', 'VDAYR',
    'AGE', 'SEX', 'RACEUN', 'ETHUN', 'IMMEDR', 'PAINSCALE',
    'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT',
] if c in eda_df.columns]

retro_cols = [c for c in [
    'WAITTIME', 'YEAR', 'PATWT', 'REGION', 'MSA',
    'ARRTIME', 'ARRIVAL_HOUR', 'VMONTH', 'VDAYR',
    'AGE', 'SEX', 'RACEUN', 'ETHUN', 'IMMEDR', 'PAINSCALE',
    'BPSYS', 'BPDIAS', 'TEMPF', 'PULSE', 'RESPR', 'POPCT',
    'LOV', 'ADMIT', 'FASTTRAK', 'OBSCLIN', 'BOARD', 'BOARDED', 'BOARDHOS',
    'BEDREG', 'IMBED', 'BEDCZAR',
    'MRI', 'XRAY', 'CT', 'CTCONTRAST', 'ULTRASOUND', 'ANYIMAGE', 'LABTEST', 'TOTPROC',
] if c in eda_df.columns]

# ── Save ───────────────────────────────────────────────────────────────────────
out = Path('../data')
eda_df.to_csv(out / 'eda_df.csv', index=False)
eda_df[arrival_cols].reset_index(drop=True).to_csv(out / 'model_A_arrival_dataset.csv', index=False)
eda_df[retro_cols].reset_index(drop=True).to_csv(out / 'model_B_retrospective_dataset.csv', index=False)

print(f'eda_df.csv                    {eda_df.shape[0]:>7} rows × {eda_df.shape[1]:>3} cols')
print(f'model_A_arrival_dataset.csv   {eda_df.shape[0]:>7} rows × {len(arrival_cols):>3} cols  (arrival features + PATWT)')
print(f'model_B_retrospective_dataset {eda_df.shape[0]:>7} rows × {len(retro_cols):>3} cols  (all features + PATWT)')

# ── Year distribution ─────────────────────────────────────────────────────────
print('\nYear distribution:')
display(
    eda_df.groupby('YEAR', dropna=False)
    .agg(rows=('WAITTIME', 'count'),
         weighted_visits=('PATWT', 'sum'),
         median_wait=('WAITTIME', 'median'))
    .reset_index()
    .assign(weighted_visits=lambda x: x['weighted_visits'].map('{:,.0f}'.format))
    .sort_values('YEAR')
)